# Basic Pitch 학습

사용 방법은 레포의 `docs/TRAINING.md`를 참고하세요. 처음에는 `SMOKE_TEST = True`로 끝까지 실행해 환경을 점검합니다.

In [ ]:
# 설정
REPO_REF = "develop"  # 학습 코드를 받을 브랜치
LOCAL_CODE = None  # 코드를 Dataset으로 올렸다면 scripts 폴더가 있는 경로 (예: "/kaggle/input/code")
SMOKE_TEST = True  # 실제 학습 시 False
MANIFEST = "/kaggle/input/<dataset-name>/manifest.csv"  # SMOKE_TEST = False일 때 사용
RESUME_FROM = None  # 이어서 학습: "/kaggle/input/<이전 결과 폴더>/run"

EPOCHS = 2 if SMOKE_TEST else 50
STEPS_PER_EPOCH = 5 if SMOKE_TEST else 100
VALIDATION_STEPS = 2 if SMOKE_TEST else 10
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
PATIENCE = 5

REPO = LOCAL_CODE or "/tmp/repo"
TRAIN_PY = "/tmp/bp/bin/python"
EXPORT_PY = "/tmp/bp-export/bin/python"
RECORDS = "/tmp/tfrecords"
RUN = "/kaggle/working/run"
if SMOKE_TEST:
    MANIFEST = "/tmp/smoke/manifest.csv"

In [ ]:
# GPU 확인과 코드 받기
!nvidia-smi
!rm -rf /tmp/repo /tmp/basic-pitch
if not LOCAL_CODE:
    !git clone --depth 1 -b "{REPO_REF}" \
      https://github.com/capstoneBU-02/capstoneBu-2026-BE.git {REPO}
!git clone --depth 1 https://github.com/spotify/basic-pitch.git /tmp/basic-pitch

In [ ]:
# 학습 환경 설치
!pip install -q uv
!uv venv /tmp/bp --python 3.10
!uv pip install --python {TRAIN_PY} "tensorflow[and-cuda]==2.15.0.post1" "numpy<2"
!uv pip install --python {TRAIN_PY} -e /tmp/basic-pitch

In [ ]:
# 데이터 변환
if SMOKE_TEST:
    !{TRAIN_PY} {REPO}/scripts/train/make_smoke_data.py --output /tmp/smoke
!rm -rf {RECORDS}
!{TRAIN_PY} {REPO}/scripts/train/prepare_data.py --manifest "{MANIFEST}" --output {RECORDS}

In [ ]:
# 학습
import pathlib
import shutil

resume_flag = ""
if RESUME_FROM:
    pathlib.Path(RUN).mkdir(parents=True, exist_ok=True)
    for name in ("last.weights.h5", "best.weights.h5", "state.json", "history.csv"):
        shutil.copy(pathlib.Path(RESUME_FROM) / name, RUN)
    resume_flag = "--resume"

!{TRAIN_PY} {REPO}/scripts/train/train.py --data {RECORDS} --output {RUN} {resume_flag} \
  --epochs {EPOCHS} --steps-per-epoch {STEPS_PER_EPOCH} --validation-steps {VALIDATION_STEPS} \
  --batch-size {BATCH_SIZE} --learning-rate {LEARNING_RATE} --patience {PATIENCE}

In [ ]:
# 변환·평가 환경 설치 (학습 환경과 분리)
!uv venv /tmp/bp-export --python 3.10
!uv pip install --python {EXPORT_PY} "tensorflow==2.15.0.post1" "numpy<2"
!uv pip install --python {EXPORT_PY} -e /tmp/basic-pitch
!uv pip install --python {EXPORT_PY} tf2onnx onnx onnxruntime

In [ ]:
# ONNX 변환과 기존 모델 비교
!{EXPORT_PY} {REPO}/scripts/train/export_onnx.py \
  --weights {RUN}/best.weights.h5 --output {RUN}/model.onnx
!{EXPORT_PY} {REPO}/scripts/train/evaluate.py --manifest "{MANIFEST}" \
  --candidate {RUN}/model.onnx --output {RUN}/evaluation.csv